In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-22 01:36:47,236]::2022985491::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint file is kept.
# Whenever a better validation MSE is found across all seeds,
# this same file is overwritten.
from vit import TimeSformer


best_model_path = "./save_timesformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        # Re-create DataLoaders for each seed so that the
        # shuffled training order is also controlled by seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Every seed starts from a newly initialized model.
        # ----------------------------------------------------
        model=TimeSformer()#num_frames=frame_nums,batchsz = batchsz
        model.to(device)

        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for audio, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                audio = audio.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot,audio)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for audio, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()
                    audio = audio.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot,audio)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            # Convert to Python float for comparisons / logging.
            current_val_mse = val_avg_loss.item()

            # ------------------------------------------------
            # Best result of the CURRENT seed
            # ------------------------------------------------
            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds.
            #
            # Only this ONE file is kept. If a new global best
            # is found, the same file is overwritten.
            # ------------------------------------------------
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mse,
                }, best_model_path)

                logger.info(
                    "Global best updated | seed: {} | epoch: {} | MSE: {:.6f}".format(
                        seed, i + 1, current_val_mse
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | MSE: {:.6f} | "
                "O-MSE: {:.6f} | C-MSE: {:.6f} | E-MSE: {:.6f} | "
                "A-MSE: {:.6f} | N-MSE: {:.6f}".format(
                    seed, i + 1, current_val_mse,
                    val_avg_o_loss.item(), val_avg_c_loss.item(),
                    val_avg_e_loss.item(), val_avg_a_loss.item(),
                    val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping for the CURRENT seed.
            # Stop when validation MSE has not improved for
            # 20 consecutive epochs.
            # ------------------------------------------------
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | best epoch: {} | best MSE: {:.6f}".format(
                        seed, i + 1, seed_best_epoch, seed_best_mse
                    )
                )
                break

        # ----------------------------------------------------
        # Store only scalar results for this seed.
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mse": seed_best_mse
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MSE = {seed_best_mse:.6f}"
        )

        # ----------------------------------------------------
        # Release memory before starting the next seed.
        # ----------------------------------------------------
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary
# ============================================================

all_mse = np.array(
    [result["best_mse"] for result in seed_results],
    dtype=np.float64
)

mean_mse = all_mse.mean()
std_mse = all_mse.std(ddof=1) if len(all_mse) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | MSE {:.6f}".format(
            result["seed"], result["best_epoch"], result["best_mse"]
        )
    )

print("-" * 80)
print(f"Mean MSE : {mean_mse:.6f}")
print(f"Std  MSE : {std_mse:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"MSE={global_best_mse:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 42 | Train 1/120: 100%|██████████| 1091/1091 [05:10<00:00,  3.51it/s]
Seed 42 | Valid 1/120: 100%|██████████| 191/191 [00:22<00:00,  8.62it/s]
[2026-09-22 01:42:32,117]::1335210312::INFO::Global best updated | seed: 42 | epoch: 1 | MSE: 1.137174
[2026-09-22 01:42:32,118]::1335210312::DEBUG::Seed: 42 | Epoch: 1 | MSE: 1.137174 | O-MSE: 0.965680 | C-MSE: 1.197658 | E-MSE: 1.030156 | A-MSE: 1.256735 | N-MSE: 1.235639
Seed 42 | Train 2/120: 100%|██████████| 1091/1091 [05:11<00:00,  3.50it/s]
Seed 42 | Valid 2/120: 100%|██████████| 191/191 [00:21<00:00,  8.70it/s]
[2026-09-22 01:48:07,895]::1335210312::DEBUG::Seed: 42 | Epoch: 2 | MSE: 1.158018 | O-MSE: 1.028670 | C-MSE: 1.231399 | E-MSE: 1.148973 | A-MSE: 1.241758 | N-MSE: 1.139289
Seed 42 | Train 3/120: 100%|██████████| 1091/1091 [05:12<00:00,  3.49it/s]
Seed 42 | Valid 3/120: 100%|██████████| 191/191 [00:21<00:00,  8.75it/s]
[2026-09-22 01:53:44,168]::1335210312::DEBUG::Seed: 42 | Epoch: 3 | MSE: 1.146442 | O-MSE: 0.961839 | C-MSE: 

Seed 42 finished | Best epoch = 6 | Best MSE = 1.089108

Start training | seed = 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 123 | Train 1/120: 100%|██████████| 1091/1091 [05:15<00:00,  3.46it/s]
Seed 123 | Valid 1/120: 100%|██████████| 191/191 [00:22<00:00,  8.53it/s]
[2026-09-22 04:08:42,479]::1335210312::DEBUG::Seed: 123 | Epoch: 1 | MSE: 1.124922 | O-MSE: 0.982722 | C-MSE: 1.229476 | E-MSE: 1.136673 | A-MSE: 1.287547 | N-MSE: 0.988194
Seed 123 | Train 2/120: 100%|██████████| 1091/1091 [05:15<00:00,  3.46it/s]
Seed 123 | Valid 2/120: 100%|██████████| 191/191 [00:22<00:00,  8.53it/s]
[2026-09-22 04:14:22,881]::1335210312::DEBUG::Seed: 123 | Epoch: 2 | MSE: 1.135690 | O-MSE: 1.029060 | C-MSE: 1.325451 | E-MSE: 1.000254 | A-MSE: 1.233740 | N-MSE: 1.089946
Seed 123 | Train 3/120: 100%|██████████| 1091/1091 [05:14<00:00,  3.46it/s]
Seed 123 | Valid 3/120: 100%|██████████| 191/191 [00:22<00:00,  8.59it/s]
[2026-09-22 04:20:02,388]::1335210312::DEBUG::Seed: 123 | Epoch: 3 | MSE: 1.121991 | O-MSE: 0.902681 | C-MSE: 1.291833 | E-MSE: 1.059775 | A-MSE: 1.351844 | N-MSE: 1.003821
Seed 123 | Train 4/120: 100%|██

Seed 123 finished | Best epoch = 3 | Best MSE = 1.121991

Start training | seed = 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


...

Seed 777 finished | Best epoch = 1 | Best MSE = 1.106767

Start training | seed = 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 1004 | Train 1/120: 100%|██████████| 1091/1091 [05:15<00:00,  3.45it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 191/191 [00:23<00:00,  8.14it/s]
[2026-09-22 08:18:12,201]::1335210312::DEBUG::Seed: 1004 | Epoch: 1 | MSE: 1.183197 | O-MSE: 1.146296 | C-MSE: 1.376780 | E-MSE: 1.153873 | A-MSE: 1.253958 | N-MSE: 0.985081
Seed 1004 | Train 2/120: 100%|██████████| 1091/1091 [05:13<00:00,  3.48it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 191/191 [00:22<00:00,  8.59it/s]
[2026-09-22 08:23:50,257]::1335210312::DEBUG::Seed: 1004 | Epoch: 2 | MSE: 1.153943 | O-MSE: 0.970301 | C-MSE: 1.360189 | E-MSE: 1.138240 | A-MSE: 1.252105 | N-MSE: 1.048882
Seed 1004 | Train 3/120: 100%|██████████| 1091/1091 [05:13<00:00,  3.48it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 191/191 [00:22<00:00,  8.58it/s]
[2026-09-22 08:29:28,553]::1335210312::DEBUG::Seed: 1004 | Epoch: 3 | MSE: 1.137825 | O-MSE: 0.874716 | C-MSE: 1.249181 | E-MSE: 1.026918 | A-MSE: 1.441293 | N-MSE: 1.097018
Seed 1004 | Train 4/12

Seed 1004 finished | Best epoch = 4 | Best MSE = 1.102859

Start training | seed = 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 2024 | Train 1/120: 100%|██████████| 1091/1091 [05:10<00:00,  3.51it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 191/191 [00:22<00:00,  8.48it/s]
[2026-09-22 10:33:16,585]::1335210312::DEBUG::Seed: 2024 | Epoch: 1 | MSE: 1.194170 | O-MSE: 1.448548 | C-MSE: 1.198132 | E-MSE: 1.082178 | A-MSE: 1.211565 | N-MSE: 1.030423
Seed 2024 | Train 2/120: 100%|██████████| 1091/1091 [05:11<00:00,  3.50it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 191/191 [00:22<00:00,  8.52it/s]
[2026-09-22 10:38:53,196]::1335210312::DEBUG::Seed: 2024 | Epoch: 2 | MSE: 1.139630 | O-MSE: 0.996870 | C-MSE: 1.264299 | E-MSE: 1.027191 | A-MSE: 1.271702 | N-MSE: 1.138087
Seed 2024 | Train 3/120: 100%|██████████| 1091/1091 [05:11<00:00,  3.50it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 191/191 [00:22<00:00,  8.49it/s]
[2026-09-22 10:44:29,519]::1335210312::DEBUG::Seed: 2024 | Epoch: 3 | MSE: 1.134004 | O-MSE: 0.826703 | C-MSE: 1.341380 | E-MSE: 1.068143 | A-MSE: 1.377227 | N-MSE: 1.056567
Seed 2024 | Train 4/12

Seed 2024 finished | Best epoch = 6 | Best MSE = 1.080942

MULTI-SEED RESULTS
Seed   42 | Best Epoch   6 | MSE 1.089108
Seed  123 | Best Epoch   3 | MSE 1.121991
Seed  777 | Best Epoch   1 | MSE 1.106767
Seed 1004 | Best Epoch   4 | MSE 1.102859
Seed 2024 | Best Epoch   6 | MSE 1.080942
--------------------------------------------------------------------------------
Mean MSE : 1.100333
Std  MSE : 0.015960
Global best: seed=2024, epoch=6, MSE=1.080942
Saved checkpoint: ./save_timesformer_folder/model_best_multiseed.pth
